In [1]:
import sqlite3
import pandas as pd

# 1. Kết nối tới cơ sở dữ liệu SQLite (Tạo file database ảo 'my_data1.db')
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

# 2. Tải tập dữ liệu SpaceX chuẩn của IBM phục vụ cho phần SQL
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(url)

# Làm sạch tên cột để tránh các ký tự đặc biệt gây lỗi SQL (ví dụ: Landing _Outcome thành Landing_Outcome)
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('(', '')
df.columns = df.columns.str.replace(')', '')

# Lưu DataFrame vào bảng có tên là SPACEXTBL trong Database SQL
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")
print("Đã tải dữ liệu và khởi tạo bảng 'SPACEXTBL' thành công!")

# Hàm phụ trợ để chạy câu lệnh SQL và hiển thị kết quả bằng Pandas DataFrame cho đẹp mắt
def run_query(query):
    return pd.read_sql_query(query, con)


# --- THỰC HIỆN 10 CÂU TRUY VẤN SQL CỦA BÀI CAPSTONE ---

print("\n=== TASK 1: Hiển thị danh sách các bệ phóng (Launch Site) độc nhất ===")
q1 = "SELECT DISTINCT Launch_Site FROM SPACEXTBL;"
print(run_query(q1))

print("\n=== TASK 2: Hiển thị 5 bản ghi phóng tên lửa có bệ phóng bắt đầu bằng ký tự 'CCA' ===")
q2 = "SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;"
print(run_query(q2))

print("\n=== TASK 3: Tính tổng khối lượng Payload được vận chuyển bởi các tên lửa phóng cho NASA (CRS) ===")
q3 = "SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_NASA FROM SPACEXTBL WHERE Customer = 'NASA (CRS)';"
print(run_query(q3))

print("\n=== TASK 4: Tính khối lượng Payload trung bình được vận chuyển bởi phiên bản tên lửa 'F9 v1.1' ===")
q4 = "SELECT AVG(PAYLOAD_MASS__KG_) AS Average_Payload_Mass_F9v11 FROM SPACEXTBL WHERE Booster_Version = 'F9 v1.1';"
print(run_query(q4))

print("\n=== TASK 5: Tìm ngày đầu tiên phóng và hạ cánh thành công trên bệ phóng mặt đất (Success (ground pad)) ===")
q5 = "SELECT MIN(Date) AS First_Successful_Ground_Pad_Landing FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)';"
print(run_query(q5))

print("\n=== TASK 6: Liệt kê các tên lửa (Booster_Version) hạ cánh thành công bằng tàu không người lái (Success (drone ship)) và có khối lượng hàng hóa từ 4000 đến 6000 kg ===")
q6 = """
SELECT Booster_Version 
FROM SPACEXTBL 
WHERE Landing_Outcome = 'Success (drone ship)' 
AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;
"""
print(run_query(q6))

print("\n=== TASK 7: Thống kê tổng số lượng các kết quả nhiệm vụ phóng thành công (Success) và thất bại (Failure) ===")
q7 = "SELECT Mission_Outcome, COUNT(*) AS Total_Count FROM SPACEXTBL GROUP BY Mission_Outcome;"
print(run_query(q7))

print("\n=== TASK 8: Liệt kê các phiên bản tên lửa mang khối lượng hàng hóa lớn nhất (MAX Payload Mass) ===")
q8 = """
SELECT Booster_Version, PAYLOAD_MASS__KG_ 
FROM SPACEXTBL 
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);
"""
print(run_query(q8))

print("\n=== TASK 9: Liệt kê kết quả hạ cánh thất bại bằng tàu (Failure (drone ship)), phiên bản tên lửa, bệ phóng trong năm 2015 ===")
q9 = """
SELECT SUBSTR(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site 
FROM SPACEXTBL 
WHERE Landing_Outcome = 'Failure (drone ship)' 
AND SUBSTR(Date, 1, 4) = '2015';
"""
print(run_query(q9))

print("\n=== TASK 10: Xếp hạng số lượng kết quả hạ cánh (Landing_Outcome) trong khoảng thời gian từ '2010-06-04' đến '2017-03-20' theo thứ tự giảm dần ===")
q10 = """
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count 
FROM SPACEXTBL 
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' 
GROUP BY Landing_Outcome 
ORDER BY Outcome_Count DESC;
"""
print(run_query(q10))

# Đóng kết nối cơ sở dữ liệu
con.close()

Đã tải dữ liệu và khởi tạo bảng 'SPACEXTBL' thành công!

=== TASK 1: Hiển thị danh sách các bệ phóng (Launch Site) độc nhất ===
    Launch_Site
0   CCAFS LC-40
1   VAFB SLC-4E
2    KSC LC-39A
3  CCAFS SLC-40

=== TASK 2: Hiển thị 5 bản ghi phóng tên lửa có bệ phóng bắt đầu bằng ký tự 'CCA' ===
         Date  Time_UTC Booster_Version  Launch_Site  \
0  2010-06-04  18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1  2010-12-08  15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2  2012-05-22   7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3  2012-10-08   0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4  2013-03-01  15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  PAYLOAD_MASS__KG_  \
0               Dragon Spacecraft Qualification Unit                  0   
1  Dragon demo flight C1, two CubeSats, barrel of...                  0   
2                              Dragon demo flight C2                525   
3                                       SpaceX CRS-1                